## Importanto o dataset

In [13]:
import seaborn as sns
import requests, zipfile, io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [14]:
url = "https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/master.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(".")  # cria a pasta Fake.br-Corpus-master/

In [15]:
os.listdir(".")

['04_RF_training.ipynb',
 'Fake.br-Corpus-master',
 '03_SVM_training.ipynb',
 '01_Data_Prep.ipynb',
 '02_LR_training.ipynb']

# Organizando colunas

In [16]:
import pandas as pd
import glob, os

def carregar_textos(pasta, label):
    registros = []
    for caminho in sorted(glob.glob(os.path.join(pasta, "*.txt"))):
        repWith = ""
        if (label == 0):
            repWith = "t"
        id_noticia = os.path.basename(caminho).replace(".txt", repWith)
        with open(caminho, encoding="utf-8") as f:
            texto = f.read()
        registros.append({"id": id_noticia, "texto": texto, "label": label})
    return pd.DataFrame(registros)

df_fake = carregar_textos("Fake.br-Corpus-master/full_texts/fake", label=1)
df_true = carregar_textos("Fake.br-Corpus-master/full_texts/true", label=0)
df_textos = pd.concat([df_fake, df_true], ignore_index=True)

colunas_meta = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "num_tokens",
    "num_palavras",
    "num_types",
    "num_links",
    "num_maiusculas",
    "num_verbos",
    "num_verbos_subj_imp",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_verbos_modais",
    "num_pron_1_2_sing",
    "num_pron_1_plural",
    "num_pronomes",
    "pausalidade",
    "num_caracteres",
    "tam_medio_sentenca",
    "tam_medio_palavra",
    "pct_erros_ortograficos",
    "emotividade",
    "diversidade"
]


def carregar_metadados(pasta, label):
    registros = []
    for caminho in sorted(glob.glob(os.path.join(pasta, "*.txt"))):
        repWith = ""
        if (label == 0):
          repWith = "t"
        id_noticia = os.path.basename(caminho).replace("-meta.txt", repWith)
        with open(caminho, encoding="utf-8") as f:
            valores = [l.strip() for l in f.readlines()]
        registro = dict(zip(colunas_meta, valores))
        registro["id"], registro["label"] = id_noticia, label
        registros.append(registro)
    return pd.DataFrame(registros)

df_meta = pd.concat([
    carregar_metadados("Fake.br-Corpus-master/full_texts/fake-meta-information", 1),
    carregar_metadados("Fake.br-Corpus-master/full_texts/true-meta-information", 0),
], ignore_index=True)

df = df_textos.merge(df_meta.drop(columns="label"), on="id")

# Truncar texto

In [17]:
def truncar(texto, n=200):
    return " ".join(str(texto).split()[:n])

df["texto_trunc"] = df["texto"].apply(truncar)

# Tratamento dos metadados

## Conversão de string para numero

In [18]:
colunas_numericas = colunas_meta[4:]
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors="coerce")

## Conversão de números abslutos para relativo a cada 100 palavras

In [19]:
num_pal = df["num_palavras"]

In [20]:
df["taxa_maiusculas"] = df["num_maiusculas"] / num_pal * 100
df["taxa_verbos"] = df["num_verbos"] / num_pal * 100
df['taxa_verbos_subj_imp'] = df["num_verbos_subj_imp"] / num_pal * 100
df['taxa_adjetivos'] = df["num_adjetivos"] / num_pal * 100
df['taxa_adverbios'] = df["num_adverbios"] / num_pal * 100
df['taxa_verbos_modais'] = df["num_verbos_modais"] / num_pal * 100
df['taxa_pron_1_2_sing'] = df["num_pron_1_2_sing"] / num_pal * 100
df['taxa_pron_1_plural'] = df["num_pron_1_plural"] / num_pal * 100
df['taxa_pronomes'] = df["num_pronomes"] / num_pal * 100
df['taxa_substantivos'] = df["num_substantivos"] / num_pal * 100

In [21]:
def mattr(tokens, window_size=50):

    n = len(tokens)
    if n == 0:
        return 0.0
    if n <= window_size:
        return len(set(tokens)) / n  # cai pro TTR simples

    ratios = [
        len(set(tokens[i:i + window_size])) / window_size
        for i in range(n - window_size + 1)
    ]
    return sum(ratios) / len(ratios)


def calcular_diversidade(texto_trunc, window_size=100):
    tokens = str(texto_trunc).split()
    return mattr(tokens, window_size=window_size)


df["diversidade_relativa"] = df["texto_trunc"].apply(calcular_diversidade)


In [22]:
metadados = [
    "taxa_maiusculas",
    "taxa_verbos",
    "taxa_verbos_subj_imp",
    "taxa_substantivos",
    "taxa_adjetivos",
    "taxa_adverbios",
    "taxa_verbos_modais",
    "taxa_pron_1_2_sing",
    "taxa_pron_1_plural",
    "taxa_pronomes",
    "pausalidade",
    "pct_erros_ortograficos",
    "tam_medio_sentenca",
    "tam_medio_palavra",
    "diversidade_relativa",
]

In [23]:
df = df[["texto_trunc"] + metadados + ["label"]].copy()

# Preparação para o treinamento dos modelos

## Split de treino e teste

In [24]:
from sklearn.model_selection import train_test_split

# Separação entre Dados e Labels
X = df[["texto_trunc"] + metadados]
y = df["label"]

# Divisão entre split de treino e split teste
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Imports

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

## Pipelines

In [26]:
txt_word = ("txt_word", TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b[^\d\W]{2,}\b"), "texto_trunc")
txt_char = ("txt_char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True), "texto_trunc")
meta     = ("meta", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), metadados)

configs = {
    "word":            [txt_word],
    "word+char":       [txt_word, txt_char],
    "word+meta":       [txt_word, meta],
    "word+char+meta":  [txt_word, txt_char, meta],
}

In [27]:
import joblib

In [30]:
joblib.dump((X_tr, X_te, y_tr, y_te), './dados_preparados.pkl')
joblib.dump(configs, './transformers.pkl')

['./transformers.pkl']

In [ ]:
break

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

ct_word = ColumnTransformer([txt_word])
X_word_tfidf = ct_word.fit_transform(X_tr)

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_word_tfidf)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=y_tr, cmap="coolwarm", alpha=0.5, s=10)
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.title("Projeção 2D do TF-IDF (word) — fake vs real")
plt.colorbar(label="Classe")
plt.show()